Bag of Words with Cosine Similarity

Implemented by: Mr. Sudharsan

In [1]:
from google.colab import files

uploaded = files.upload()

Saving amazon_products_with_images (1).csv to amazon_products_with_images (1).csv


In [10]:

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


df = pd.read_csv("/content/amazon_products_with_images (1).csv")


df = df.fillna("")

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

df = df.dropna(subset=[
    "product_name",
    "category",
    "brand",
    "description",
    "price",
    "rating"
])

df.reset_index(drop=True, inplace=True)


df["combined_features"] = (
    df["product_name"].astype(str) + " " +
    df["brand"].astype(str) + " " +
    df["category"].astype(str) + " " +
    df["description"].astype(str)
)


bow = CountVectorizer(stop_words="english")

bow_matrix = bow.fit_transform(df["combined_features"])


cosine_sim = cosine_similarity(bow_matrix)


def recommend_products(product_name, top_n=5):

    matches = df[
        df["product_name"].str.contains(
            product_name,
            case=False,
            na=False
        )
    ]

    if matches.empty:
        print("Product not found")
        return None

    idx = matches.index[0]

    target_price = df.loc[idx, "price"]
    target_rating = df.loc[idx, "rating"]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i, score in similarity_scores:

        if i == idx:
            continue

        price = df.loc[i, "price"]
        rating = df.loc[i, "rating"]

        if abs(price - target_price) <= target_price * 0.20:

            if abs(rating - target_rating) <= 0.3:

                recommendations.append({
                    "Product": df.loc[i, "product_name"],
                    "Brand": df.loc[i, "brand"],
                    "Category": df.loc[i, "category"],
                    "Price": price,
                    "Rating": rating,
                    "Similarity Score": round(score, 3)
                })

        if len(recommendations) >= top_n:
            break

    if len(recommendations) == 0:
        print("No similar products found.")
        return None

    return pd.DataFrame(recommendations)


if __name__ == "__main__":

    product = input("Enter Product Name: ")

    result = recommend_products(product)

    if result is not None:
        print("\nRecommended Products\n")
        print(result)

Enter Product Name: Samsung Galaxy M13

Recommended Products

                                             Product    Brand    Category  \
0  Samsung Galaxy M13 (Stardust Brown, 6GB, 128GB...  Samsung  Smartphone   
1  Samsung Galaxy M13 (Aqua Green, 4GB, 64GB Stor...  Samsung  Smartphone   
2  Samsung Galaxy M13 (Midnight Blue, 4GB, 64GB S...  Samsung  Smartphone   
3  Samsung Galaxy M13 5G (Aqua Green, 6GB, 128GB ...  Samsung  Smartphone   
4  Samsung Galaxy M13 5G (Aqua Green, 6GB, 128GB ...  Samsung  Smartphone   

     Price  Rating  Similarity Score  
0  12999.0     4.1             0.986  
1  10999.0     4.1             0.957  
2  10999.0     4.1             0.943  
3  13999.0     4.1             0.832  
4  13999.0     4.1             0.832  
